# Easy Callbacks

Some methods of `Problem` are not called by the search by default, but
can be easily incorporated into `BranchAndBound` callbacks:

- `Problem.primal_heuristic`: returns a feasible solution obtained
  from a node. It is used by calling `BranchAndBound.primal_heuristic(node)`,
  which sets it as the new incumbent if it improves the current one.
- `Problem.stronger_bound`: returns a lower bound stronger (and
  usually more expensive) than `calc_bound`. It is used by calling
  `BranchAndBound.upgrade_bound(node)`, which updates the bound of the
  node if it is greater than the current one.

The available callbacks of `BranchAndBound` are `pre_eval_callback` and
`post_eval_callback` (before and after the evaluation of a node bound),
`enqueue_callback` (before a node is enqueued), `dequeue_callback`
(after a node is dequeued) and `solution_callback` (after a new
incumbent is set).

In this notebook we illustrate them with a binary knapsack problem.

In [1]:
import random

from bnbpy import BranchAndBound, Node, Problem

## Knapsack Problem

Items $i \in N$ have values $v_{i}$ and weights $w_{i}$, and a subset of
them with maximum total value must be selected without exceeding the
capacity $W$. As `bnbpy` minimizes, the cost of a solution is minus its
value.

Each node decides whether the next item is selected or not. Its lower
bound by `calc_bound` is weak: it assumes all remaining items fit in the
knapsack.

In [2]:
class Knapsack(Problem):
    def __init__(self, values: list[float], weights: list[float], capacity: float):
        super().__init__()
        self.values = values
        self.weights = weights
        self.capacity = capacity
        # Decisions for the first items: True if selected
        self.selected: list[bool] = []

    @property
    def value(self) -> float:
        return sum(v for v, s in zip(self.values, self.selected) if s)

    @property
    def weight(self) -> float:
        return sum(w for w, s in zip(self.weights, self.selected) if s)

    def calc_bound(self) -> float:
        # All remaining items fit
        k = len(self.selected)
        return -(self.value + sum(self.values[k:]))

    def is_feasible(self) -> bool:
        return len(self.selected) == len(self.values)

    def branch(self) -> list['Knapsack']:
        k = len(self.selected)
        children = []
        for take in (True, False):
            if take and self.weight + self.weights[k] > self.capacity:
                continue
            child = self.child_copy(deep=False)
            child.selected = self.selected + [take]
            children.append(child)
        return children

In [3]:
rng = random.Random(7)
n_items = 20
values = [rng.randint(10, 60) for _ in range(n_items)]
weights = [rng.randint(5, 40) for _ in range(n_items)]
capacity = sum(weights) / 3

bnb = BranchAndBound(Knapsack(values, weights, capacity))
sol = bnb.solve()
print(f"{sol} | {bnb.explored} nodes explored")

Status: OPTIMAL | Cost: -341.0 | LB: -341.0 | 18329 nodes explored


## Bound Improvements

A stronger bound is given by the linear relaxation (Dantzig bound):
remaining items are added by decreasing value per weight, and the first
one that does not fit is taken fractionally. Assume it is expensive, so
it is only computed for nodes that cannot be pruned by the regular bound,
calling `BranchAndBound.upgrade_bound` in `post_eval_callback`.

In [4]:
class KnapsackStrongerBound(Knapsack):
    def stronger_bound(self) -> float:
        k = len(self.selected)
        free = self.capacity - self.weight
        value = self.value
        order = sorted(
            range(k, len(self.values)),
            key=lambda i: self.values[i] / self.weights[i],
            reverse=True,
        )
        for i in order:
            if self.weights[i] <= free:
                free -= self.weights[i]
                value += self.values[i]
            else:
                value += self.values[i] * free / self.weights[i]
                break
        return -value


class UpgradeBoundBnB(BranchAndBound):
    def post_eval_callback(self, node: Node) -> None:
        # Only worth it if the node cannot be pruned yet
        # (other criteria could be the depth of the node, etc.)
        if node.lb < self.ub:
            self.upgrade_bound(node)

In [5]:
bnb = UpgradeBoundBnB(KnapsackStrongerBound(values, weights, capacity))
sol = bnb.solve()
print(f"{sol} | {bnb.explored} nodes explored")

Status: OPTIMAL | Cost: -341.0 | LB: -341.0 | 790 nodes explored


## Primal Heuristics

A primal heuristic completes the partial solution of a node greedily:
each remaining item is selected if it fits. It must return a new
problem (here, a `child_copy`) instead of modifying the current one.

It can be called during the search, for instance from
`post_eval_callback`, which here only calls it in the first levels of
the tree.

In [6]:
class KnapsackHeuristic(KnapsackStrongerBound):
    def primal_heuristic(self) -> 'KnapsackHeuristic':
        child = self.child_copy(deep=False)
        selected = list(self.selected)
        free = self.capacity - self.weight
        for i in range(len(selected), len(self.values)):
            take = self.weights[i] <= free
            if take:
                free -= self.weights[i]
            selected.append(take)
        child.selected = selected
        return child


class PrimalSearchBnB(UpgradeBoundBnB):
    def post_eval_callback(self, node: Node) -> None:
        if node.level < 5:
            self.primal_heuristic(node)
        super().post_eval_callback(node)

In [7]:
bnb = PrimalSearchBnB(KnapsackHeuristic(values, weights, capacity))
sol = bnb.solve()
print(f"{sol} | {bnb.explored} nodes explored")

Status: OPTIMAL | Cost: -341.0 | LB: -341.0 | 268 nodes explored


A heuristic can also be applied every time a new solution is found,
e.g., a local search that tries to improve it:

```python
class SolutionImprovementBnB(BranchAndBound):
    def solution_callback(self, node: Node) -> None:
        self.primal_heuristic(node)
```

In the Permutation Flow Shop example (`bnbprob.pafssp.CallbackBnB`),
a local search is applied every time a new incumbent is found. In the
machine deadline problem with a Lagrangian relaxation
(`bnbprob.machdeadline.DeadlineLagrangianSearch`), it is called after
the evaluation of every node, returning a solution by Smith's rule
that is also used in bound computations.